# Toybox in-context rotation pilot — one question, ONE call to `openai/gpt-6-luna`

The question (11 recorded Toybox frames: 3 before/after demonstrations, 1 query, 4 candidates) is already built and
stored in the repository under `experiments/toybox_rotation_pilot/question/`. This notebook only:

1. makes sure the repository is up to date;
2. shows the composite (what the question looks like; the composite itself is **not** sent);
3. makes **exactly one** API call: the 11 images in order, each with a neutral label, then the prompt. The script
   refuses to call again once `results/response.json` exists, and a failed call is saved and not retried;
4. shows the raw response and the comparison with the intended answer (read from `metadata.json` only after the call);
5. downloads the results.

Needs the Colab secrets `GITHUB_TOKEN` and `OPENROUTER_API_KEY` (never printed).


In [ ]:
# --- 2. Bootstrap: locate or clone the PRIVATE repository (credentials are never printed or stored) ---
import base64, importlib.util, os, pathlib, subprocess, sys

REPO_URL = "https://github.com/zhexuandliu/Decompositionality.git"
REPO_BRANCH = "main"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

def _find_repo():
    candidates = [pathlib.Path("/content/Decompositionality"), pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    for p in candidates:
        if (p / "src" / "decompositionality").is_dir():
            return p.resolve()
    return None

REPO_DIR = _find_repo()
if REPO_DIR is None:
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
    if not token:
        import getpass
        token = getpass.getpass("GitHub token with read access to the private repo (input hidden): ")
    # Pass the credential through environment-scoped git config (not the URL, not the command line,
    # not persisted in .git/config).
    _hdr = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
    _env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="1",
                GIT_CONFIG_KEY_0="http.https://github.com/.extraheader", GIT_CONFIG_VALUE_0=_hdr)
    REPO_DIR = pathlib.Path("/content/Decompositionality" if IN_COLAB else "Decompositionality").resolve()
    _r = subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
                        env=_env, capture_output=True, text=True)
    del token, _hdr, _env
    if _r.returncode != 0:
        raise RuntimeError("git clone failed (check the GITHUB_TOKEN secret / repo access):\n" + _r.stderr[-2000:])

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
for _mod in ("numpy", "pandas", "matplotlib", "torch"):
    if importlib.util.find_spec(_mod) is None:  # Colab already ships these; never reinstall torch/CUDA here
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _mod], check=True)
REPO_COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Repository:", REPO_DIR, "| commit:", REPO_COMMIT or "unknown")


In [ ]:
# --- 3. Update the repository (the question files were added recently) -------------------------------------------
import base64, subprocess
_tok = None
if IN_COLAB:
    from google.colab import userdata
    _tok = userdata.get("GITHUB_TOKEN")
_env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
if _tok:
    _env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                GIT_CONFIG_VALUE_0="AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{_tok}".encode()).decode())
_r = subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"], env=_env, capture_output=True, text=True)
del _tok, _env
print(_r.stdout[-500:] or _r.stderr[-500:])
PILOT = REPO_DIR / "experiments" / "toybox_rotation_pilot"
assert (PILOT / "question" / "inputs.json").exists(), "question files missing: pull failed?"
print("question folder:", PILOT / "question")


In [ ]:
# --- 4. Look at the question (composite = neutral labels only; not sent to the model) -------------------------------
from IPython.display import Image as _Img, display
display(_Img(filename=str(PILOT / "question" / "composite.jpg"), width=1100))
print((PILOT / "question" / "prompt.txt").read_text())


In [ ]:
# --- 5. THE single evaluation call (refuses to run again once results/response.json exists) -------------------------
if IN_COLAB and not os.environ.get("OPENROUTER_API_KEY"):
    from google.colab import userdata
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")      # never printed
r = subprocess.run([sys.executable, str(PILOT / "eval_once.py")], capture_output=True, text=True, cwd=str(PILOT))
print(r.stdout[-4000:]); print(r.stderr[-2000:])


In [ ]:
# --- 6. Raw response and download ---------------------------------------------------------------------------------
import json, shutil
resp = json.loads((PILOT / "results" / "response.json").read_text())
print("model returned:", resp.get("model_returned"), "| finish:", resp.get("finish_reason"), "| error:", resp.get("error"))
print("raw response:", repr(resp.get("raw_response")))
print("usage:", resp.get("usage"))
print("\nsummary:", (PILOT / "results" / "result_summary.json").read_text())
if IN_COLAB:
    from google.colab import files
    shutil.make_archive("/content/toybox_pilot_results", "zip", PILOT / "results")
    files.download("/content/toybox_pilot_results.zip")
